# 嵌入向量的演进思路

## 向量

### 独热编码

独热编码（One-Hot Encoding）的核心是将 $N$ 个离散类别转换为长度为 $N$ 的二进制向量，每个向量中**仅对应类别的位置为 1，其余位置全为 0**。

---

#### 具体转换 3 步走

以分类“水果”为例，假设数据集中有 3 种水果：**苹果、香蕉、橙子**。

1. **提取唯一类别**：统计所有不重复的离散值，确定总类别数 $N = 3$。
2. **分配固定席位**：为每个类别分配一个唯一的索引位置。
* 苹果 $\to$ 第 1 位
* 香蕉 $\to$ 第 2 位
* 橙子 $\to$ 第 3 位


3. **按位赋值**：遇到对应类别填 `1`，其他位置补 `0`。

---

#### 转换效果对比

| 原始输入 | 向量维度 `[苹果, 香蕉, 橙子]` | 编码结果 |
| --- | --- | --- |
| **苹果** | `[1, 0, 0]` | **1 0 0** |
| **香蕉** | `[0, 1, 0]` | **0 1 0** |
| **橙子** | `[0, 0, 1]` | **0 0 1** |
| **苹果** | `[1, 0, 0]` | **1 0 0** |

---

#### 为什么这样做？

* **消除伪数值大小关系**：如果用标签编码（苹果=1，香蕉=2，橙子=3），算法容易误以为“橙子 > 香蕉 > 苹果”或“苹果 + 香蕉 = 橙子”。独热编码让每个类别在空间中相互正交、距离均等。
* **代价是维度增加**：如果特征有 1000 个不同取值，就会凭空生成 1000 列极其稀疏的矩阵（高维稀疏性）。

#### 缺点

- 信息密度极低：虽为多维向量，但仅有 1 位有效（其余全为 0），单字表征的表达效率低下。

- 维度爆炸与高维稀疏：向量维度与类别总数严格呈 1:1 绑定；词表越大，空间维度越高，内存占用与零值计算冗余剧增。

- 孤立正交，缺乏语义关系：任意两向量内积均为 0，在几何空间中相互垂直，无法度量相似度（如“猫”和“狗”的距离等同于“猫”和“飞机”）。

- 静态硬编码，无法端到端训练：属于固定的离散规则映射，不含可学习参数，无法根据下游任务动态微调表征。

> 在机器学习和数学中，向量的维度就是这个向量包含的元素个数（长度）。

### 嵌入向量

嵌入就是将高维稀疏向量转换为稠密多维向量的结果，每个维度代表对象的某种特征

> 我们可以将数学家和物理学家投射为向量，每个属性(如是否可以跑步、是否喜欢喝咖啡等)都对应一个维度。由于数学家和物理学家在这些属性上相似，那么我们可以在这些维度赋予较高分数。

**好的嵌入向量应具备以下特征**

- 唯一性：词与嵌入向量之间是一一对应的关系，避免重复表示。
- 可区分性：词义相近的词在嵌入空间中距离较近，不同词义的词距离较远。
- 可计算性：无论词如何构成，都应在同一空间内可计算，且计算结果也在同一空间，例如可以通过简单数学计算“连接”两个向量表示新词
- 蕴含深层语义：好的嵌入向量应技能建模单词使用的复杂特征，又能建模这些用法在语境中的变化

![](../src/transformer-6-3.jpg)

| 维度 | 余弦相似度 | 点积 |
| --- | --- | --- |
| **关注核心** | **纯粹的“方向 / 主题一致性”** | **“方向吻合度”与“强度 / 重要性”的复合共鸣** |
| **尺度敏感性** | 尺度无关（Scale-invariant） | 尺度高度敏感（Scale-sensitive） |
| **典型直觉** | **内容相关性**：短文与长文讲的是同一个主题，则余弦为 1。 | **物理做功 / 能量共振**：推力越大且位移越大，做功越大；若推力为 0，即使方向一致做功也为 0。 |

**余弦相似度的深层本质：特征比例的纯粹性**

* 余弦相似度只关心**各个维度的相对比例**，而不关心**绝对数值**。
* 例如向量 $A = [1, 2]$ 和向量 $B = [100, 200]$，余弦相似度为 $1.0$。在文本检索中，这代表一篇文章只提了 1 次“苹果、香蕉”，另一篇提了 100 次“苹果、香蕉”，它们在主题分布上是完全一致的，余弦相似度排除了文本长度对相似度的干扰。

**点积的深层本质：特征相关性与重要性的加权组合**

* 点积在保留方向相关性的同时，**赋予高幅度特征更高的权重**。
* **推荐系统（协同过滤）**：用户隐向量 $\vec{u}$ 与物品隐向量 $\vec{v}$ 的点积 $\vec{u} \cdot \vec{v}$。若用户偏好极其强烈（模长大），物品热度/特征明显（模长大），两者的共振点积就会显著放大；如果某个用户是冷启动用户（模长接近 0），点积自然趋于 0。
* **Transformer 的自注意力机制（$QK^T$）**：Query 和 Key 计算点积时，模型通过学习模长来表达“这个 Token 的重要度与置信度”。高权重的 Token 可以主动放大自身模长，在点积竞争中获得更大的注意力分数。

### 三、 优缺点对比

#### 余弦相似度

* **优点**：
* **消除尺度干扰**：对向量整体放大或缩放不敏感，适合长度不一的文本、音频或直方图数据。
* **数值有界可解释**：范围固定在 $[-1, 1]$，无需考虑数据范围即可轻松设定相似度阈值（如 $> 0.8$ 即判定为相似）。


* **缺点**：
* **完全抹杀幅度信息**：无法区分“高频重要”和“低频微弱”的特征差异。
* **计算开销略大**：每次计算都需要计算模长（涉及平方、求和、开根号），在超大规模暴力检索时比纯点积慢。

#### 点积

* **优点**：
* **计算极其廉价**：仅包含乘加操作（FMA），能够直接利用 CPU/GPU 的底层 GEMM 矩阵乘法加速。
* **信息保留更完整**：能够兼顾“匹配度”与“置信度/活跃度”。


* **缺点**：
* **数值无界**：点积结果大小取决于向量本身的模长，无法直接设定固定的截断阈值。
* **易受长尾尺度污染**：若未做归一化，一个在方向上完全不相关的极长向量，可能因为某些维度的绝对值过大而产生比强相关短向量更高的点积得分。


# 嵌入层

嵌入层的主要功能是将文本中词汇的数字表示（token序列）转化为高维向量表示（即词嵌入），从而在高维空间中捕捉词汇之间的关系

## 词嵌入的转换过程

① 通过梯度下降学习到更有意义的词向量表示，构成嵌入矩阵
② 从嵌入矩阵中查找到token对应的嵌入向量，即将token从原始维度 $d_{vocab}$ 投射到自定义维度 $d_{model}$, $d_{model}$的大小也决定了模型的最终参数量

## 嵌入层的实现

### 嵌入矩阵

嵌入层是一个矩阵，或者说是一个与token ID一起工作的线性层，用于将输入和输出的token ID转换为$d_{model}$维度的向量。这个矩阵是一个由词表衍生的查找表，存储了模型词表中每个token的向量表示。嵌入矩阵是一个大小为VxD的二维矩阵，其中V是词表大小，D是嵌入维度。例如，若词表大小为V=10000，词向量维度D=300，则嵌入矩阵的大小为10000x300,每个词表中的词都有一个300维的向量表示

![](../src/transformer-6-5.jpg)

从另一个角度来看，token序号可以被视为独热编码，本质上是用独热编码乘以嵌入矩阵


### 哈佛源码

《The Annotated Transformer》是哈佛大学 NLP 团队发布的《Attention Is All You Need》逐行复现教程，用 PyTorch 把论文中每一条公式都翻译成可运行的源码，是学习 Transformer 实现最经典的材料。

> 原文出处：<https://nlp.seas.harvard.edu/annotated-transformer/>

下面先给出其中的**嵌入层完整源码**，再逐行拆解，最后用 4 个小实验亲手验证它的每一个设计。

In [1]:
# =====================================================================
# 哈佛《The Annotated Transformer》源码：嵌入层（Embeddings）
# =====================================================================
import math
import torch
import torch.nn as nn


class Embeddings(nn.Module):
    """把 token ID 序列映射为 d_model 维的稠密向量序列。

    参数：
        d_model : 模型隐藏维度（论文中为 512），决定每个 token 被映射成多长的向量
        vocab   : 词表大小，即模型认识的 token 总数（嵌入矩阵的行数）
    """

    def __init__(self, d_model, vocab):
        super(Embeddings, self).__init__()
        # 查找表（lookup table）：本质是一个形状为 (vocab, d_model) 的可学习矩阵，
        # 词表中每个 token 独占一行，这一行就是它的"词向量"
        self.lut = nn.Embedding(vocab, d_model)
        # 记录模型维度，供 forward 中缩放使用
        self.d_model = d_model

    def forward(self, x):
        # ① 查表：把每个 token ID 替换为嵌入矩阵中对应的那一行
        #    形状变化：(batch, seq_len) --> (batch, seq_len, d_model)
        # ② 缩放：乘以 sqrt(d_model)，把嵌入的数值量级拉回与 d_model 解耦的水平（下文详细解释）
        return self.lut(x) * math.sqrt(self.d_model)

#### 逐行拆解：整个类只有 3 个关键点

##### 关键点 1：`nn.Embedding(vocab, d_model)` —— 嵌入矩阵就是一张查找表

`nn.Embedding` 内部维护一个形状为 `(vocab, d_model)` 的**可学习参数矩阵** $E$。以论文配置为例（$vocab \approx 37000$，$d_{model}=512$），矩阵有 37000 行、512 列。它的前向计算非常"朴素"：**把 token ID 当作行号，直接返回矩阵的对应行**

$$\text{Embedding}(x) = E[x]$$

它与"用线性层做投影"的对比：

| 对比项 | `nn.Embedding(vocab, d_model)` | `nn.Linear(vocab, d_model)` |
| --- | --- | --- |
| 输入要求 | 整数 token ID | 独热编码后的浮点向量 |
| 数学本质 | 按行号索引取行 $E[x]$ | 独热编码 × 权重矩阵 $\text{onehot}(x)W$ |
| 计算方式 | 直接索引（查表） | 完整矩阵乘法 |
| 参数形状 | `(vocab, d_model)` | `(d_model, vocab)`（转置） |
| 每词计算量 | $O(d_{model})$ | $O(vocab \times d_{model})$ |

两者在数学上**完全等价**（实验 2 验证），但查表跳过了与零元素相乘的巨大浪费——这正是嵌入层用查表而非矩阵乘法实现的原因。

##### 关键点 2：`self.lut(x)` —— 形状的跳跃

forward 的输入是整数 token ID 序列，输出是浮点向量序列，每个整数被"展开"成 $d_{model}$ 维向量：

$$\underbrace{(batch,\ seq\_len)}_{\text{整数 ID}} \longrightarrow \underbrace{(batch,\ seq\_len,\ d_{model})}_{\text{稠密向量}}$$

从这一刻起，序列中每个位置都拥有了**连续、可学习、能表达语义**的表示。

##### 关键点 3：`* math.sqrt(self.d_model)` —— 论文中最"神秘"的一行

查表结束后还要整体乘上 $\sqrt{d_{model}}$（$d_{model}=512$ 时约为 22.6 倍）。为什么要放大？这是嵌入层唯一"看不出必要性"的操作，也是初学者最常困惑的地方——下一节用实验揭示它的动机。

In [6]:
# =====================================================================
# 实验 1：实例化嵌入层，亲眼看一次"查表"过程
# =====================================================================
torch.manual_seed(42)

# 用极小规模方便观察：词表 10 个 token，每个映射为 8 维向量
d_model, vocab_size = 8, 10
embedding = Embeddings(d_model, vocab_size)

# 一批 token ID：2 个"句子"，每句 4 个 token
token_ids = torch.tensor([
    [0, 3, 5, 9],
    [1, 2, 7, 4],
])

print(f"嵌入矩阵（查找表）形状 : {tuple(embedding.lut.weight.shape)}          -> (vocab, d_model)")
print(f"输入 token ID 形状     : {tuple(token_ids.shape)}                 -> (batch, seq_len)")

output = embedding(token_ids)
print(f"输出嵌入向量形状       : {tuple(output.shape)}              -> (batch, seq_len, d_model)")

print("-" * 62)
# token 5 的词向量就是嵌入矩阵的第 5 行
print("嵌入矩阵第 5 行（token 5 的原始词向量）:")
print(embedding.lut.weight[5])

# 输出中第 1 个句子的第 3 个位置正是 token 5（注意已乘 sqrt(d_model)）
print("\n前向传播输出 output[0, 2]（= 原始词向量 x sqrt(d_model)）:")
print(output[0, 2])

print("\n验证 output[0, 2] == weight[5] * sqrt(d_model) ->",
      torch.allclose(output[0, 2], embedding.lut.weight[5] * math.sqrt(d_model)))

嵌入矩阵（查找表）形状 : (10, 8)          -> (vocab, d_model)
输入 token ID 形状     : (2, 4)                 -> (batch, seq_len)
输出嵌入向量形状       : (2, 4, 8)              -> (batch, seq_len, d_model)
--------------------------------------------------------------
嵌入矩阵第 5 行（token 5 的原始词向量）:
tensor([-1.5576,  0.9956, -0.8798, -0.6011, -1.2742,  2.1228, -1.2347, -0.4879],
       grad_fn=<SelectBackward0>)

前向传播输出 output[0, 2]（= 原始词向量 x sqrt(d_model)）:
tensor([-4.4055,  2.8161, -2.4884, -1.7003, -3.6038,  6.0041, -3.4921, -1.3800],
       grad_fn=<SelectBackward0>)

验证 output[0, 2] == weight[5] * sqrt(d_model) -> True


#### 实验 2：验证"查表"等价于"独热编码 × 嵌入矩阵"

回顾前文的结论：token ID 本质是独热编码的**压缩存储**。数学上，查表与"用独热编码左乘嵌入矩阵"完全等价——独热向量中只有第 $x$ 位是 1，乘矩阵 $E$ 的结果恰好是"取出 $E$ 的第 $x$ 行"：

$$\text{onehot}(x) \cdot E = E[x]$$

工程上选择索引而非真的构造独热编码，一举两得：**省内存**（不必生成巨大的独热张量）、**省计算**（跳过所有乘零运算）。下面的代码严格验证等价性，并量化计算量差距。

In [3]:
# =====================================================================
# 实验 2：nn.Embedding 查表 vs 独热编码 × 矩阵
# =====================================================================
import torch.nn.functional as F

# 方式一：嵌入层的真实做法 —— 直接用 ID 索引取行
by_lookup = embedding.lut(token_ids)                            # (2, 4, 8)

# 方式二：数学本质 —— 先构造独热编码，再做矩阵乘法
one_hot = F.one_hot(token_ids, num_classes=vocab_size).float()  # (2, 4, 10)
by_matmul = one_hot @ embedding.lut.weight                      # (2,4,10) @ (10,8) -> (2, 4, 8)

print(f"独热编码形状: {tuple(one_hot.shape)}，每个 token 的 10 维向量中仅 1 个 1（90% 是 0）")
print(f"两种方式结果是否完全一致: {torch.allclose(by_lookup, by_matmul)}")

batch, seq_len = token_ids.shape
print(f"\n本例计算量: 查表 {batch * seq_len} 次索引  vs  矩阵乘法 {batch * seq_len * vocab_size} 次乘加")
print("真实场景 vocab=30000 时，矩阵乘法每个词要做 30000 次乘加，其中 99.997% 都在与 0 相乘")

独热编码形状: (2, 4, 10)，每个 token 的 10 维向量中仅 1 个 1（90% 是 0）
两种方式结果是否完全一致: True

本例计算量: 查表 8 次索引  vs  矩阵乘法 80 次乘加
真实场景 vocab=30000 时，矩阵乘法每个词要做 30000 次乘加，其中 99.997% 都在与 0 相乘


#### 关键设计：为什么 `forward` 里要乘 $\sqrt{d_{model}}$

论文 §3.4 原文：

> In the embedding layers, we multiply those sums by $\sqrt{d_{model}}$.

这一乘服务于 Transformer 的三个全局设计约束。

##### 动机一：让嵌入的"音量"与 $d_{model}$ 解耦

嵌入矩阵通常采用**小方差初始化**（原始 TensorFlow 实现中每维方差约 $1/d_{model}$，即标准差 $1/\sqrt{d_{model}}$），这样查出的词向量整体模长约 1，与 $d_{model}$ 无关。乘上 $\sqrt{d_{model}}$ 后，每维方差恰好回到 1：

$$\underbrace{\frac{1}{\sqrt{d_{model}}}}_{\text{初始化的标准差}} \times \underbrace{\sqrt{d_{model}}}_{\text{缩放}} = 1$$

无论 $d_{model}=512$ 还是 $2048$，嵌入信号都稳定在"每维标准差 ≈ 1"的量级——学习率、初始化等超参数不必随模型宽度重新调整。

##### 动机二：与位置编码的量级匹配

嵌入层的下一步是与**正弦位置编码逐元素相加**。正弦编码每个分量取值在 $[-1, 1]$、整体模长约为 $\sqrt{d_{model}/2}$。若嵌入不缩放（每维 std 仅 $1/\sqrt{d_{model}}$），相加时位置信号会**完全淹没词义信息**；乘 $\sqrt{d_{model}}$ 后两者处于同一量级，"词义 + 位置"才能均衡融合。

##### 动机三：残差流的"第一棒"

Transformer 主干是一条由残差连接串起的信号之河：$x + \text{Sublayer}(x)$ 层层累加。嵌入向量是这条河的**源头**，必须与后续子层输出（LayerNorm 之后约为每维 std ≈ 1）量级对齐，否则源头信号要么被冲淡、要么迫使后续层重新适应尺度。

> **常见疑惑**：PyTorch 的 `nn.Embedding` 默认用 $\mathcal{N}(0,1)$ 初始化，每维方差本来就是 1，此时再乘 $\sqrt{d_{model}}$ 反而把模长放大 $\sqrt{d_{model}}$ 倍（$d_{model}=512$ 时约 22.6 倍）。这是复现者最常踩的坑——**论文的缩放是配合"小方差初始化"约定使用的**。下面的实验用小方差初始化复现论文设定。

In [4]:
# =====================================================================
# 实验 3：乘 sqrt(d_model) 前后，嵌入量级如何随 d_model 变化
# =====================================================================
print(f"{'d_model':>8} | {'未缩放 每维std':>14} | {'未缩放 向量模长':>15} | {'缩放后 每维std':>14} | {'缩放后 向量模长':>15}")
print("-" * 82)

for d in [64, 512, 2048]:
    torch.manual_seed(0)
    lut = nn.Embedding(1000, d)
    # 复现论文式小方差初始化：每维标准差 = 1/sqrt(d_model)
    nn.init.normal_(lut.weight, mean=0.0, std=d ** -0.5)

    ids = torch.randint(0, 1000, (1, 100))   # 随机抽取 100 个 token
    raw = lut(ids)[0]                        # (100, d) 未缩放
    scaled = raw * math.sqrt(d)              # 缩放后

    print(f"{d:>8} | {raw.std():>14.3f} | {raw.norm(dim=-1).mean():>16.3f} "
          f"| {scaled.std():>14.3f} | {scaled.norm(dim=-1).mean():>16.3f}")

print("\n观察：")
print("① 未缩放时每维 std = 1/sqrt(d)：d 越大信号越弱（0.125 -> 0.022），位置编码会彻底淹没词义")
print("② 缩放后每维 std 稳定在 1.0 附近 —— 嵌入量级与 d_model 彻底解耦")
print("③ 缩放后模长约为 sqrt(d)，恰好与正弦位置编码（模长约为 sqrt(d/2) ~ sqrt(d)）处于同一量级")

 d_model |      未缩放 每维std |        未缩放 向量模长 |      缩放后 每维std |        缩放后 向量模长
----------------------------------------------------------------------------------
      64 |          0.126 |            1.007 |          1.011 |            8.056
     512 |          0.044 |            0.999 |          0.999 |           22.594
    2048 |          0.022 |            1.001 |          1.001 |           45.283

观察：
① 未缩放时每维 std = 1/sqrt(d)：d 越大信号越弱（0.125 -> 0.022），位置编码会彻底淹没词义
② 缩放后每维 std 稳定在 1.0 附近 —— 嵌入量级与 d_model 彻底解耦
③ 缩放后模长约为 sqrt(d)，恰好与正弦位置编码（模长约为 sqrt(d/2) ~ sqrt(d)）处于同一量级


## 嵌入的训练

> **一句话总览**：嵌入矩阵不是"造好就一劳永逸"的静态字典，而是一组**可学习参数**——先初始化，再随任务在反向传播中不断更新，最后收敛固化，让"意思相近的词"在空间里自动靠在一起。

![](../src/embedding-training-flow.svg)

### 嵌入的来源：两条路线

| 路线 | 做法 | 适用场景 | 特点 |
| --- | --- | --- | --- |
| **静态嵌入（拿来主义）** | 直接加载 Word2Vec / GloVe 预训练词向量，训练中**冻结**不动 | 监督数据较少时（冻结参数、只训下游，本身也是一种正则化） | 省算力、稳定；嵌入矩阵退化为纯查找表 |
| **动态嵌入（自己学）** | 嵌入矩阵作为模型参数的一部分，随训练**不断更新改进** | Transformer / LLM 的标准做法 | 能学到贴合任务与语料的表示 |

### 为什么要再训练？——预训练嵌入是"固化"的

- **一词一向量，不懂语境**：静态嵌入在训练完成那一刻就冻结了，"苹果"在"吃苹果"和"苹果发布会"里是同一个向量，无法结合上下文刻画局部语义。
- **通用 ≠ 适用**：静态嵌入源自通用语料，主要保留整体语义，落到具体领域（法律 / 医疗 / 代码）或具体任务上往往水土不服。
- **任务在变，表示也要跟着变**：让词向量在实际语料的训练中继续演化，才能对齐当前任务和数据。

### 让模型自己设计：从人工特征到自动学习

- 机器学习模型吃不了文本、图像这类非结构化数据，必须先转成数字。
- 深度学习之前靠**人工特征工程**：特征一多维度爆炸，而且"一个词有几十个语义属性，每个属性该赋什么值？"——人根本定不好。
- 深度学习把思路反转：**把嵌入当作模型参数**，让神经网络在训练中自动学出对任务最有用的表示，特征之间还能相互作用，无须人工设计。

### 训练流程三步走：初始化 → 更新 → 固化

#### ① 初始化：给每个词一个"出生点"

嵌入矩阵是 $V \times D$ 的二维矩阵（$V$ = 词表大小，$D$ = 向量维度），两种开局方式：

- **随机初始化**：高维空间中随机生成的向量近似正交 → 各词出生时彼此独立、互不相关（"你、我、他"出生时毫无关系）；词与词的相关性（语义、语法、文化）完全靠后续训练从语料中**学出来**。
- **预训练初始化**：用 Word2Vec / GloVe / FastText 的向量当起点，自带丰富语义关系，为后续 Transformer 层打好地基，训练中再结合语义与任务微调。

#### ② 更新：相关词拉近，无关词推远

- 学习目标源自**分布式假设**：上下文相似的词，向量应该相近。
- 训练中通过**反向传播**反复调整嵌入矩阵：常在同一上下文出现的词被**拉近**，从不搭界的词被**推远**，向量由此带上丰富的上下文含义。
- 循环迭代，向量逐渐稳定、调整幅度越来越小，直到收敛——相似词积累了相似的更新，最终在嵌入空间中聚成一团。

#### ③ 固化：收敛即得到嵌入层权重

- 网络收敛或到达停止条件时，各层参数相对固定，嵌入矩阵的权重就是最终得到的嵌入。
- 此时 token 与高维向量一一映射，查表即可单独取出任意词的向量。

> 训练完成 ≠ 一劳永逸：进入注意力机制后，每个 token 的嵌入还会持续"吸收"周围 token 的信息（无论位置远近），表示随上下文动态变化——这正是 LLM 嵌入与静态嵌入的分水岭。

### 小结：字典 vs 种子

| | Word2Vec 这类独立嵌入模型 | LLM 中的嵌入 |
| --- | --- | --- |
| **类比** | 一本**字典**：查到即终点 | 一粒**种子**：查到只是发芽 |
| **与上下文** | 无关，一词一向量 | 在注意力层中融合上下文，动态变化 |
| **通用性** | 通用、预先构建完成的最终产品 | 为具体模型 / 任务"定制"，并非通用组件 |
| **训练方式** | 独立训练、一次成型 | 与整个模型**端到端**一起优化 |
| **本质** | 静态成品 | 非静态，承载"启动动态过程"的使命 |


# 文本嵌入

> **一句话总览**：文本嵌入把一段文字压成一个定长向量（如 BGE 的 1024 维），让「语义相近的文本」在向量空间里彼此靠近——它是搜索、RAG、推荐、聚类的地基。

![](../src/text-embedding-evolution.svg)

一条演化主线：**One-hot（1986~）→ Word2Vec（2013）→ Transformer（2018~）→ BGE（2023~）**，向量从「稀疏、无语义」一路走向「稠密、上下文感知、句子级」。

## 是什么

#### ① 它解决什么问题

One-hot 只能回答「是不是这个词」——稀疏、正交、无语义，算不出「苹果 ≈ 香蕉」。文本嵌入把词 / 句映射到低维稠密向量，把语义编码进几何关系：**余弦相似度越高，语义越接近**。

#### ② 静态 vs 动态

| 阶段 | 代表 | 向量特点 | 短板 |
| --- | --- | --- | --- |
| 静态词向量 | Word2Vec / GloVe | 一词一向量，训完固定 | 一词多义没法区分 |
| 动态上下文向量 | Transformer / BERT | 同一词在不同句子里向量不同 | 直接拿来做检索效果一般 |
| 句子级嵌入 | BGE / E5 / GTE | 整句 → 定长向量（如 1024 维） | 依赖对比学习训练 |

#### ③ 嵌入模型 vs 重排序器

| | Embedding 模型 | Reranker |
| --- | --- | --- |
| 输出 | 每段文本一个向量，可离线建库 | 一对 (query, doc) 一个分数 |
| 交互 | 双塔：各自编码再算相似度 | Cross-Encoder：拼接后整体过模型 |
| 速度 | 快（ANN 向量检索） | 慢（只能精排几十条） |
| 角色 | 粗排召回 top50 | 精排出 top5 |

RAG 经典分工：**向量库召回（快而粗）→ reranker 精排（准而慢）**。

---

## Word2Vec

> **核心思想（分布式假设）**：「看一个词的邻居，就知道它的意思。」（Firth, 1957）

![](../src/word2vec-cbow-skipgram.svg)

CBOW（上下文 → 中心词）与 Skip-gram（中心词 → 上下文）互为镜像；小语料上 Skip-gram 对低频词更友好。

#### 底层实现路径（六步走）

**Step 1｜滑窗切对**：窗口大小 2 时，例句「今天 天气 很 好」产出 `(今天,天气)`、`(今天,很)`、`(天气,很)`… 等训练对。

**Step 2｜查表前向**：维护两个矩阵——

- $W \in \mathbb{R}^{V \times N}$：中心词向量表，**训练完就是嵌入，最终产物**
- $W' \in \mathbb{R}^{N \times V}$：上下文向量表，**只是脚手架，训完可丢**

中心词 one-hot 查表得向量 $h$，与上下文词向量做内积。

**Step 3｜softmax 打分**：把内积变成「这个词出现在我旁边的概率」：

$$P(w_i \mid w_c) = \frac{\exp(v_{w_i}'^{\top} v_{w_c})}{\sum_{j=1}^{V} \exp(v_{w_j}'^{\top} v_{w_c})}$$

**Step 4｜卡脖子**：分母要对全词表（几十万词）求和，每个训练对都算一遍全表 softmax——根本跑不动。

**Step 5｜负采样**：只留 1 个正例 + $k$ 个负例（按词频的 $0.75$ 次幂采样，照顾低频词），把 $V$ 分类降级成 $k+1$ 个二分类：正例拉近、负例推远——

$$L = -\log \sigma(v_{w_O}'^{\top} v_{w_c}) \; - \sum_{i=1}^{k} \log \sigma(-v_{w_i}'^{\top} v_{w_c})$$

**Step 6｜SGD 更新**：只更新正例、负例涉及的行列，其余几百万行原封不动——这正是 Word2Vec 能在百万词级语料上训练的原因。

```python
# 负采样核心循环（简化版）
for center, context in pairs:
    negatives = sample_by_freq(V, k)              # 按词频^0.75 采 k 个负例
    h = W[center]                                 # 查表前向
    loss  = -logsigmoid(h @ Wc[context])          # 正例：拉近
    loss += -logsigmoid(-h @ Wc[negatives]).sum() # 负例：推远
    loss.backward(); opt.step()                   # 只更新涉及的行
```

#### 产物与天花板

训练完，$W$ 就是词向量表。经典性质：**king − man + woman ≈ queen**，语义方向可线性组合。

但天花板也明显：**静态**（「苹果」在「吃苹果」和「苹果发布会」里是同一个向量）、**没有句向量**（只能词向量平均）、**OOV**（词表外没向量）、**换语料要重训**。

---

## BGE

> BGE（BAAI General Embedding，智源 2023）是国内最常用的开源嵌入系列。它不发明新结构，而是把「**预训练底座 → 大规模对比学习 → 精标微调**」三件事做到位。

![](../src/bge-training-pipeline.svg)

#### 底层实现路径（五步走）

**Step 1｜选底座 + 池化**：以 XLM-RoBERTa / BERT 为底座，取 **[CLS] 位置的输出**（1024 维）再 L2 归一化，之后点积即余弦相似度：

$$\text{sim}(q,d) = \frac{h_q \cdot h_d}{\|h_q\| \, \|h_d\|}$$

**Step 2｜准备正负样本**：正样本对来自 (query, 正确文档) 配对数据；**难负例**用 BM25 或旧模型检索「看起来相关其实不相关」的文档，逼模型在更细粒度上学会分辨。

**Step 3｜对比学习**：batch 内其他样本的文档全部当负例（in-batch negatives，一个 batch 免费凑出几千个负例），损失用 InfoNCE（温度 $\tau = 0.01$，低温让模型更挑剔）：

$$\mathcal{L} = -\log \frac{\exp(\text{sim}(q, d^{+})/\tau)}{\exp(\text{sim}(q, d^{+})/\tau) + \sum_{d^{-}} \exp(\text{sim}(q, d^{-})/\tau)}$$

**Step 4｜RetroMAE 预训练**（对比学习之前，先给底座打检索底子）：**编轻解狠**——编码器只轻遮 15~30%，单层解码器却要狠遮重建全文（50~70%）。重建难度倒逼编码器把全句语义压进一个向量。

**Step 5｜精标微调**：人工标注的高质量 (query, doc) 对 + 难负例微调，并给 query 加指令前缀「为这个句子生成表示以用于检索相关文章：」——指令告诉模型此刻在为检索服务。

#### 怎么用、怎么读分数

```python
from FlagEmbedding import FlagModel
model = FlagModel('BAAI/bge-large-zh-v1.5',
                  query_instruction_for_retrieval='为这个句子生成表示以用于检索相关文章：')
q = model.encode_queries('什么是文本嵌入？')            # 查询侧要加指令
d = model.encode(['文本嵌入把文字映射成向量', '今天天气很好'])
scores = q @ d.T                                       # 归一化后点积 = 余弦相似度
```

BGE 相似度通常落在 [0.6, 1] 区间，**绝对值意义不大，看相对排序**。

#### BGE-M3：一个模型，三种表示

| 表示 | 原理 | 擅长 |
| --- | --- | --- |
| Dense | [CLS] 向量 + ANN 检索 | 语义泛化 |
| Sparse | 可学习稀疏权重（类 BM25） | 关键词精确匹配 |
| Multi-Vector | ColBERT 式 token 级晚交互 | 长文档精排 |

三种分数经自蒸馏（self-knowledge distillation）融合，支持 100+ 语言、最长 8192 token。

---

## Word2Vec vs BGE

| 维度 | Word2Vec | BGE |
| --- | --- | --- |
| 年代 | 2013 | 2023 |
| 粒度 | 词 | 句 / 段 |
| 向量 | 静态 | 动态（上下文感知） |
| 结构 | 浅层查表 + 负采样 | Transformer 底座 + 对比学习 |
| 训练目标 | 猜邻居 | 语义对齐（InfoNCE） |
| 维度 | 100~300 | 1024+ |
| 句向量 | 不支持（需平均） | 原生支持 |
| OOV | 无向量 | 子词切分可覆盖 |
| 迁移 | 换语料需重训 | 零样本 / 轻量微调 |
| 典型用途 | 词类比、轻量特征 | 检索、RAG、聚类、去重 |

> **带走五句话**
>
> 1. 嵌入的本质：把语义变成几何，「近」=「意思近」。
> 2. Word2Vec 用「猜邻居」这个代理任务，第一次学出可迁移的语义空间。
> 3. 静态向量的天花板是一词多义，Transformer 的动态编码解决它。
> 4. BGE 的胜负手不在结构，在**数据 + 损失函数**：难负例 + 低温 InfoNCE + 指令微调。
> 5. RAG 分工：embedding 召回（快而粗），reranker 精排（准而慢）。

**参考**

- Mikolov et al., *Distributed Representations of Words and Phrases and their Compositionality*, 2013（负采样与 CBOW/Skip-gram，arXiv:1310.4546）
- Xiao et al., *C-Pack: Packaged Resources To Advance General Chinese Embedding*, SIGIR 2024（BGE，arXiv:2309.07597）
- *RetroMAE: Pre-training Retrieval-oriented Language Models With Minimal Auto-encoding*, 2022（arXiv:2205.12035）